# Exogenous Model AB Test

이 notebook은 `patchtst_exo`, `timexer`, `exotst`를 같은 DSIO exogenous 데이터에서 비교하기 위한 빠른 실험용 템플릿입니다.

이 notebook에서 확인할 수 있는 것:
- `future_exo_source='columns'` 와 `future_exo_source='callback'` 계약 차이
- 모델별 train/inference batch shape
- 소규모 샘플에 대한 학습 + 추론 + metric 계산
- GT와 예측값을 aggregate / part-level plot으로 함께 비교

권장 사용 순서:
1. Setup / Config 셀 실행
2. Data Preparation 셀 실행
3. Loader Smoke Test 셀로 shape 확인
4. 필요 시 `models_to_run`, `sample_part_count`, epoch 설정을 줄인 뒤 Tiny AB Run 셀 실행


In [ ]:
from __future__ import annotations

%load_ext autoreload
%autoreload 2

import argparse
import sys
from pathlib import Path

import polars as pl
from IPython.display import Image, display


import importlib.util
import json
import os
import random
import sys
from pathlib import Path

import numpy as np
import polars as pl
import torch


# Remote kernel이면 여기에 서버 기준 repo 절대경로를 넣을 수 있습니다.
# Example: REPO_ROOT_OVERRIDE = Path("/home/ubuntu/ts_forecaster_lib")
# repo clone이 서버에 없고 modeling_module만 설치되어 있어도 notebook는 동작하도록 구성합니다.
REPO_ROOT_OVERRIDE = None


def _looks_like_repo(path: Path) -> bool:
    return (path / "pyproject.toml").exists() and (path / "src").exists()


def _iter_named_repo_candidates(root: Path, repo_name: str = "ts_forecaster_lib", max_depth: int = 4):
    if not root.exists() or not root.is_dir():
        return

    try:
        root_resolved = root.resolve()
    except Exception:
        root_resolved = root

    stack = [(root_resolved, 0)]
    while stack:
        current, depth = stack.pop()
        if current.name == repo_name:
            yield current
        if depth >= max_depth:
            continue
        try:
            children = list(current.iterdir())
        except Exception:
            continue
        for child in children:
            if child.is_dir() and not child.name.startswith('.'):
                stack.append((child, depth + 1))


def find_repo_root(start: Path, explicit_repo_root: Path | None = None) -> Path | None:
    env_repo_root = os.environ.get("TS_FORECASTER_REPO_ROOT")

    candidates = []
    if explicit_repo_root is not None:
        candidates.append(Path(explicit_repo_root).expanduser().resolve())
    if env_repo_root:
        candidates.append(Path(env_repo_root).expanduser().resolve())
    candidates.extend([start, *start.parents])

    home = Path.home()
    common_roots = [
        home,
        home / "workspace",
        home / "workspaces",
        home / "projects",
        home / "PycharmProjects",
        Path("/workspace"),
        Path("/workspaces"),
        Path("/home"),
        Path("/root"),
    ]
    for root in common_roots:
        candidates.extend(_iter_named_repo_candidates(root))

    seen = set()
    for candidate in candidates:
        if candidate in seen:
            continue
        seen.add(candidate)
        if _looks_like_repo(candidate):
            return candidate

    return None


def resolve_import_paths() -> tuple[Path | None, Path | None]:
    repo_root = find_repo_root(Path.cwd().resolve(), explicit_repo_root=REPO_ROOT_OVERRIDE)
    src_root = repo_root / "src" if repo_root is not None else None

    if src_root is not None and str(src_root) not in sys.path:
        sys.path.insert(0, str(src_root))

    spec = importlib.util.find_spec("modeling_module")
    if spec is None or spec.origin is None:
        raise RuntimeError(
            "Could not import modeling_module. Either set REPO_ROOT_OVERRIDE to the server repo path, "
            "set TS_FORECASTER_REPO_ROOT, or install the package on the remote environment."
        )

    module_init = Path(spec.origin).resolve()
    module_root = module_init.parent

    if repo_root is None and module_root.parent.name == "src":
        repo_root = module_root.parent.parent
        src_root = repo_root / "src"

    return repo_root, src_root


NOTEBOOK_DIR = Path.cwd().resolve()
REPO_ROOT, SRC_ROOT = resolve_import_paths()

from model_test.exogenous_test.exogenous_ab_utils import (
    attach_actuals,
    build_callback_future_exo_components,
    compute_metric_tables,
    ensure_dir,
    make_point_forecast_result_table,
    plot_latest_revision_aggregate,
    plot_latest_revision_part_grid,
    plot_metric_summary,
    resolve_single_plan_week,
    select_eval_ids_with_full_actual_coverage,
    select_latest_revision,
    select_plot_parts,
)
from model_test.exogenous_test.run_exogenous_model_ab import (
    CALLBACK_LOOKUP_FUTURE_COLS,
    DATE_COL,
    EXO_SOURCE_FALLBACK,
    FUTURE_EXO_CONT_COLS,
    FREQ,
    ID_COL,
    MODEL_SPECS,
    PAST_EXO_CAT_COLS,
    PAST_EXO_CONT_COLS,
    TARGET_EXO_SOURCE,
    TARGET_SOURCE,
    Y_COL,
    build_architecture_config,
    build_exo_data_request,
    resolve_model_specs,
)
from model_test.total_train.dsio_total_running import (
    configure_torch_runtime,
    load_polars_table,
    prepare_exo_one_table,
    prepare_target_df,
    resolve_exo_source,
    set_global_seed,
)
from modeling_module import (
    ArtifactConfig,
    RuntimeConfig,
    SSLConfig,
    TrainRequest,
    TrainerConfig,
    load_predictor,
    DataColumnConfig,
    DataRequest,
    DataWindowConfig,
    ExogenousConfig,
    LoaderConfig,

    train,
)
from modeling_module.api.data import build_datamodule

print('REPO_ROOT :', REPO_ROOT)
print('SRC_ROOT  :', SRC_ROOT)


## Config

아래 값을 바꾸면서 빠르게 실험할 수 있습니다.

- `future_exo_source='columns'`: `future_exo_cont_cols` 사용
- `future_exo_source='callback'`: `future_exo_cb + part_future_exo_fn` 사용
- `models_to_run`: 전체 비교 또는 일부 모델만 선택
- `sample_part_count`: 빠른 smoke test면 작게, 본격 비교면 크게


In [ ]:
artifact_root = REPO_ROOT / 'artifacts' / 'exogenous_test' / 'notebook_ab'
ensure_dir(artifact_root)

models_to_run = ['patchtst_exo', 'timexer', 'exotst']
future_exo_source = 'columns'  # 'columns' or 'callback'

lookback = 52
horizon = 26
# backtest는 명시한 한 주차(plan_week)에 대해서만 수행합니다.
plan_week = 202541
plot_part_count = 20
sample_part_count = 1000
max_parts_per_plan = 10_000

train_batch_size = 256
infer_batch_size = 128
num_workers = 0
pin_memory = False
persistent_workers = False
prefetch_factor = 2
seed = 42

# lifecycle feature 옵션
use_lifecycle_future_features = True
use_lifecycle_past_features = True
lifecycle_start_mode = 'sales_master'  # 'sales_master' | 'first_positive' | 'first_observed' | 'sales_start_col'
lifecycle_start_col = 'sales_start_yyyyww'
lifecycle_peak_weeks = 104
lifecycle_tail_weeks = 104
show_lifecycle_profile = False
lifecycle_oper_part_path = REPO_ROOT / 'raw_data' / 'raw' / 'tb_mst_oper_part.parquet'
lifecycle_sales_parts_path = REPO_ROOT / 'raw_data' / 'raw' / 'tb_dyn_sales_parts.parquet'

# intermittent weighting은 간헐 수요 비중이 높은 데이터에 유리합니다.
# 이미 intermittent part를 제거한 비교 실험이라면 False/False가 보통 더 공정합니다.
use_intermittent = False
val_use_weights = False

# 아래 epoch 값은 최소 smoke test보다 조금 더 의미 있는 비교를 위한 기본값입니다.
# 아주 빠른 확인만 원하면 1 / 1 로 다시 낮춰도 됩니다.
warmup_epochs = 50
spike_epochs = 10
lr = 1e-3
ssl_mode = 'full'
ssl_pretrain_epochs = 10
ssl_mask_ratio = 0.3
ssl_loss_type = 'mse'

set_global_seed(seed)
default_device, device_note = configure_torch_runtime()
device = default_device

print('models_to_run                  :', models_to_run)
print('future_exo_source             :', future_exo_source)
print('sample_part_count             :', sample_part_count)
print('device                        :', device)
print('use_lifecycle_future_features :', use_lifecycle_future_features)
print('use_lifecycle_past_features   :', use_lifecycle_past_features)
print('use_intermittent              :', use_intermittent)
print('val_use_weights               :', val_use_weights)
if use_lifecycle_future_features or use_lifecycle_past_features:
    print('lifecycle_start_mode          :', lifecycle_start_mode)
    print('lifecycle_start_col           :', lifecycle_start_col)
    print('lifecycle_peak_weeks          :', lifecycle_peak_weeks)
    print('lifecycle_tail_weeks          :', lifecycle_tail_weeks)
    print('lifecycle_oper_part_path      :', lifecycle_oper_part_path)
    print('lifecycle_sales_parts_path    :', lifecycle_sales_parts_path)
if device_note:
    print('device_note                   :', device_note)


## Data Preparation

이 셀은 target / exogenous parquet를 읽고, 샘플링한 뒤, train용 one-table과 plan week를 준비합니다.

`future_exo_source='callback'`이면 여기서 callback 구성요소도 같이 만듭니다.

In [ ]:
from model_test.model_test_utils import yyyyww_to_monday
import math


def yyyyww_to_week_index(yyyyww: int) -> int:
    return yyyyww_to_monday(int(yyyyww)).toordinal() // 7


def load_lifecycle_master_frames(selected_ids: list[str]) -> tuple[pl.DataFrame, pl.DataFrame]:
    oper_path = Path(lifecycle_oper_part_path)
    sales_path = Path(lifecycle_sales_parts_path)

    if not oper_path.exists():
        raise FileNotFoundError(f'lifecycle oper-part parquet not found: {oper_path}')
    if not sales_path.exists():
        raise FileNotFoundError(f'lifecycle sales-parts parquet not found: {sales_path}')

    oper_part_df = (
        pl.read_parquet(oper_path)
        .filter(pl.col(ID_COL).cast(pl.String).is_in(selected_ids))
        .select([
            pl.col(ID_COL).cast(pl.String).alias(ID_COL),
            pl.col('demand_start_dt').cast(pl.Int64),
            pl.col('demand_end_dt').cast(pl.Int64),
            pl.col('warranty').cast(pl.Int64),
        ])
        .unique(subset=[ID_COL])
        .with_columns(
            (pl.col('warranty') * 52 / 12).round(0).cast(pl.Int64).alias('warranty_weeks')
        )
    )

    sales_part_df = (
        pl.read_parquet(sales_path)
        .filter(pl.col(ID_COL).cast(pl.String).is_in(selected_ids))
        .filter(pl.col('sales_qty') > 0)
        .filter(pl.col('sales_dt').cast(pl.Int64) < int(plan_week))
        .group_by(pl.col(ID_COL).cast(pl.String).alias(ID_COL))
        .agg([
            pl.col('sales_dt').cast(pl.Int64).min().alias('first_sales_week'),
        ])
        .sort(ID_COL)
    )

    return oper_part_df, sales_part_df


def build_lifecycle_anchor_df(
    target_df: pl.DataFrame,
    exo_df: pl.DataFrame,
    oper_part_df: pl.DataFrame,
    sales_part_df: pl.DataFrame,
    *,
    start_mode: str,
    start_col: str,
) -> pl.DataFrame:
    first_observed = (
        target_df.group_by(ID_COL)
        .agg(pl.col(DATE_COL).min().cast(pl.Int64).alias('first_observed_week'))
    )

    first_positive = (
        target_df
        .filter((pl.col(Y_COL) > 0) & (pl.col(DATE_COL).cast(pl.Int64) < int(plan_week)))
        .group_by(ID_COL)
        .agg(pl.col(DATE_COL).min().cast(pl.Int64).alias('first_positive_week'))
    )

    start_col_df = None
    resolved_mode = str(start_mode)
    if resolved_mode == 'sales_start_col' and start_col in exo_df.columns:
        start_col_df = (
            exo_df.select([pl.col(ID_COL).cast(pl.String).alias(ID_COL), pl.col(start_col).cast(pl.Int64).alias('sales_start_week')])
            .drop_nulls(['sales_start_week'])
            .group_by(ID_COL)
            .agg(pl.col('sales_start_week').min().alias('sales_start_week'))
        )
    elif resolved_mode == 'sales_start_col':
        resolved_mode = 'sales_master'

    anchor_df = (
        first_observed
        .join(first_positive, on=ID_COL, how='left')
        .join(oper_part_df, on=ID_COL, how='left')
        .join(sales_part_df, on=ID_COL, how='left')
    )
    if start_col_df is not None:
        anchor_df = anchor_df.join(start_col_df, on=ID_COL, how='left')

    if resolved_mode == 'first_observed':
        start_expr = pl.col('first_observed_week')
    elif resolved_mode == 'first_positive':
        start_expr = pl.coalesce([pl.col('first_positive_week'), pl.col('first_observed_week')])
    elif resolved_mode == 'sales_master':
        start_expr = pl.coalesce([
            pl.col('demand_start_dt'),
            pl.col('first_sales_week'),
            pl.col('first_observed_week'),
        ])
    elif resolved_mode == 'sales_start_col':
        start_expr = pl.coalesce([
            pl.col('sales_start_week'),
            pl.col('demand_start_dt'),
            pl.col('first_sales_week'),
            pl.col('first_observed_week'),
        ])
    else:
        raise ValueError(
            f'Unsupported lifecycle_start_mode={start_mode!r}. '
            'Use one of: sales_master | first_positive | first_observed | sales_start_col'
        )

    return (
        anchor_df
        .with_columns([
            start_expr.cast(pl.Int64).alias('lifecycle_start_week'),
            pl.col('demand_end_dt').cast(pl.Int64).alias('lifecycle_end_week'),
        ])
        .select([
            ID_COL,
            'first_observed_week',
            'first_positive_week',
            'first_sales_week',
            'demand_start_dt',
            'demand_end_dt',
            'warranty_weeks',
            'lifecycle_start_week',
            'lifecycle_end_week',
        ])
    )


def build_lifecycle_feature_frame(
    target_df: pl.DataFrame,
    exo_df: pl.DataFrame,
    oper_part_df: pl.DataFrame,
    sales_part_df: pl.DataFrame,
    *,
    start_mode: str,
    start_col: str,
    peak_weeks: int,
    tail_weeks: int,
) -> pl.DataFrame:
    anchor_df = build_lifecycle_anchor_df(
        target_df,
        exo_df,
        oper_part_df,
        sales_part_df,
        start_mode=start_mode,
        start_col=start_col,
    )

    peak = float(max(int(peak_weeks), 1))
    tail = float(max(int(tail_weeks), 1))

    lifecycle = (
        target_df.select([pl.col(ID_COL).cast(pl.String).alias(ID_COL), pl.col(DATE_COL).cast(pl.Int64).alias(DATE_COL)])
        .unique()
        .join(anchor_df, on=ID_COL, how='left')
        .with_columns([
            pl.col(DATE_COL).map_elements(lambda x: int(yyyyww_to_week_index(int(x))), return_dtype=pl.Int64).alias('week_idx'),
            pl.when(pl.col('lifecycle_start_week').is_not_null())
            .then(pl.col('lifecycle_start_week').map_elements(lambda x: int(yyyyww_to_week_index(int(x))), return_dtype=pl.Int64))
            .otherwise(None)
            .alias('lifecycle_start_idx'),
            pl.when(pl.col('lifecycle_end_week').is_not_null())
            .then(pl.col('lifecycle_end_week').map_elements(lambda x: int(yyyyww_to_week_index(int(x))), return_dtype=pl.Int64))
            .otherwise(None)
            .alias('lifecycle_end_idx'),
        ])
        .with_columns([
            (pl.col('week_idx') - pl.col('lifecycle_start_idx')).alias('lc_elapsed_raw'),
            (pl.col('lifecycle_end_idx') - pl.col('week_idx')).alias('lc_weeks_to_end_raw'),
            (pl.col('lifecycle_end_idx') - pl.col('lifecycle_start_idx')).alias('lc_total_cycle_weeks'),
        ])
        .with_columns([
            pl.col('lc_elapsed_raw').clip(lower_bound=0).alias('lc_age_week'),
            (pl.col('lc_elapsed_raw') < 0).cast(pl.Float64).alias('lc_before_start_flag'),
            (pl.col('lc_weeks_to_end_raw') < 0).cast(pl.Float64).alias('lc_after_demand_end_flag'),
            ((pl.col('lc_elapsed_raw') >= 0) & (pl.col('lc_weeks_to_end_raw') >= 0)).cast(pl.Float64).alias('lc_in_active_window'),
        ])
        .with_columns([
            (pl.col('lc_age_week') / peak).clip(0.0, 1.0).alias('lc_age_norm_peak'),
            (pl.col('lc_age_week') / (peak + tail)).clip(0.0, 1.0).alias('lc_age_norm_total'),
            (pl.col('lc_age_week') >= peak).cast(pl.Float64).alias('lc_after_peak_flag'),
            ((pl.col('lc_age_week') - peak).clip(lower_bound=0) / tail).clip(0.0, 1.0).alias('lc_decay_norm_tail'),
            pl.col('lc_age_week').map_elements(lambda x: float(math.log1p(max(float(x), 0.0))), return_dtype=pl.Float64).alias('lc_age_log1p'),
        ])
        .with_columns([
            pl.when(pl.col('lc_total_cycle_weeks') > 0)
            .then((pl.col('lc_elapsed_raw') / pl.col('lc_total_cycle_weeks')).clip(0.0, 1.0))
            .otherwise(0.0)
            .alias('lc_demand_progress'),
            pl.col('lc_weeks_to_end_raw').clip(lower_bound=0).alias('lc_weeks_to_end_pos'),
        ])
        .with_columns([
            (pl.col('lc_weeks_to_end_pos') / tail).clip(0.0, 1.0).alias('lc_weeks_to_end_norm'),
            pl.when(pl.col('warranty_weeks') > 0)
            .then((pl.col('lc_age_week') / pl.col('warranty_weeks')).clip(0.0, 1.0))
            .otherwise(0.0)
            .alias('lc_warranty_progress'),
            pl.when(pl.col('warranty_weeks') > 0)
            .then(((pl.col('warranty_weeks') - pl.col('lc_age_week')).clip(lower_bound=0) / pl.col('warranty_weeks')).clip(0.0, 1.0))
            .otherwise(0.0)
            .alias('lc_weeks_to_warranty_end_norm'),
            pl.when(pl.col('warranty_weeks') > 0)
            .then((pl.col('lc_age_week') > pl.col('warranty_weeks')).cast(pl.Float64))
            .otherwise(0.0)
            .alias('lc_after_warranty_flag'),
        ])
        .select([
            ID_COL,
            DATE_COL,
            'lc_age_norm_peak',
            'lc_age_norm_total',
            'lc_after_peak_flag',
            'lc_decay_norm_tail',
            'lc_age_log1p',
            'lc_demand_progress',
            'lc_weeks_to_end_norm',
            'lc_after_demand_end_flag',
            'lc_before_start_flag',
            'lc_in_active_window',
            'lc_warranty_progress',
            'lc_weeks_to_warranty_end_norm',
            'lc_after_warranty_flag',
        ])
        .sort([ID_COL, DATE_COL])
    )
    return lifecycle


def build_lifecycle_augmented_exo(exo_df: pl.DataFrame, lifecycle_df: pl.DataFrame, lifecycle_cols: list[str]) -> pl.DataFrame:
    return (
        exo_df.join(lifecycle_df, on=[ID_COL, DATE_COL], how='left')
        .with_columns([pl.col(c).fill_null(0.0).cast(pl.Float64) for c in lifecycle_cols])
    )


target_raw = load_polars_table(TARGET_SOURCE, 'tb_master_target')

target_df = prepare_target_df(
    target_raw,
    id_col=ID_COL,
    date_col=DATE_COL,
    y_col=Y_COL,
    use_id_sample=False,
    max_ids=sample_part_count,
    sample_part_count=sample_part_count,
    min_obs=lookback + horizon,
    seed=seed,
)

exo_source = resolve_exo_source(TARGET_EXO_SOURCE, EXO_SOURCE_FALLBACK)
exo_raw = load_polars_table(exo_source, exo_source.name)
selected_ids = target_df.select(pl.col(ID_COL).cast(pl.String)).unique().sort(ID_COL).get_column(ID_COL).to_list()
oper_part_df, sales_part_df = load_lifecycle_master_frames(selected_ids)

lifecycle_df = None
lifecycle_feature_cols = []
exo_raw_effective = exo_raw
future_exo_cont_cols_effective = list(FUTURE_EXO_CONT_COLS)
past_exo_cont_cols_effective = list(PAST_EXO_CONT_COLS)

if use_lifecycle_future_features and future_exo_source != 'columns':
    raise ValueError(
        'Lifecycle future feature mode currently supports only future_exo_source="columns".'
    )

if use_lifecycle_future_features or use_lifecycle_past_features:
    lifecycle_df = build_lifecycle_feature_frame(
        target_df,
        exo_raw,
        oper_part_df,
        sales_part_df,
        start_mode=lifecycle_start_mode,
        start_col=lifecycle_start_col,
        peak_weeks=lifecycle_peak_weeks,
        tail_weeks=lifecycle_tail_weeks,
    )
    lifecycle_feature_cols = [
        'lc_age_norm_peak',
        'lc_age_norm_total',
        'lc_after_peak_flag',
        'lc_decay_norm_tail',
        'lc_age_log1p',
        'lc_demand_progress',
        'lc_weeks_to_end_norm',
        'lc_after_demand_end_flag',
        'lc_before_start_flag',
        'lc_in_active_window',
        'lc_warranty_progress',
        'lc_weeks_to_warranty_end_norm',
        'lc_after_warranty_flag',
    ]
    exo_raw_effective = build_lifecycle_augmented_exo(
        exo_raw,
        lifecycle_df,
        lifecycle_feature_cols,
    )
    if use_lifecycle_future_features:
        future_exo_cont_cols_effective = list(FUTURE_EXO_CONT_COLS) + list(lifecycle_feature_cols)
    if use_lifecycle_past_features:
        past_exo_cont_cols_effective = list(PAST_EXO_CONT_COLS) + list(lifecycle_feature_cols)

exo_one_table = prepare_exo_one_table(
    target_df=target_df,
    exo_df=exo_raw_effective,
    id_col=ID_COL,
    date_col=DATE_COL,
    y_col=Y_COL,
    past_exo_cont_cols=list(past_exo_cont_cols_effective),
    future_exo_cont_cols=list(future_exo_cont_cols_effective),
    past_exo_cat_cols=list(PAST_EXO_CAT_COLS),
)

plan_week = resolve_single_plan_week(
    target_df,
    date_col=DATE_COL,
    horizon=horizon,
    plan_week=plan_week,
)
train_cutoff = int(plan_week)
train_one_table = exo_one_table.filter(pl.col(DATE_COL) < train_cutoff)
eval_ids = select_eval_ids_with_full_actual_coverage(
    target_df,
    id_col=ID_COL,
    date_col=DATE_COL,
    plan_weeks=[plan_week],
    horizon=horizon,
)
if not eval_ids:
    raise ValueError(
        'No evaluation ids have full actual coverage for the selected plan week. '
        'Move plan_week earlier or increase sample_part_count.'
    )
eval_target_df = target_df.filter(pl.col(ID_COL).cast(pl.String).is_in(eval_ids))
infer_one_table = exo_one_table.filter(pl.col(ID_COL).cast(pl.String).is_in(eval_ids))

future_exo_cb = None
part_future_exo_fn = None
if future_exo_source == 'callback':
    future_exo_cb, part_future_exo_fn = build_callback_future_exo_components(
        exo_one_table,
        id_col=ID_COL,
        date_col=DATE_COL,
        lookup_future_cols=CALLBACK_LOOKUP_FUTURE_COLS,
    )

print('target_raw shape :', target_raw.shape)
print('target_df shape  :', target_df.shape)
print('exo_raw shape    :', exo_raw.shape)
print('oper_part_df shape:', oper_part_df.shape)
print('sales_part_df shape:', sales_part_df.shape)
print('sales_part_df columns:', sales_part_df.columns)
print('exo_one_table    :', exo_one_table.shape)
print('train_one_table  :', train_one_table.shape)
print('eval_target_df   :', eval_target_df.shape)
print('infer_one_table  :', infer_one_table.shape)
print('eval_id_count    :', len(eval_ids))
print('plan_week        :', plan_week)
print('train_cutoff     :', train_cutoff)
print('past_exo_cols_effective   :', past_exo_cont_cols_effective)
print('future_exo_cols_effective :', future_exo_cont_cols_effective)
if use_lifecycle_future_features or use_lifecycle_past_features:
    print('lifecycle_df shape       :', lifecycle_df.shape)
    if show_lifecycle_profile:
        lifecycle_profile_df = (
            target_df.select([ID_COL, DATE_COL, Y_COL])
            .join(lifecycle_df, on=[ID_COL, DATE_COL], how='left')
            .with_columns(((pl.col('lc_age_log1p') * 10).floor() * 4).cast(pl.Int64).alias('age_bucket_proxy'))
            .group_by('age_bucket_proxy')
            .agg([
                pl.len().alias('n_rows'),
                pl.col(Y_COL).mean().alias('mean_qty'),
                pl.col(Y_COL).median().alias('median_qty'),
                pl.col(Y_COL).quantile(0.9).alias('p90_qty'),
            ])
            .sort('age_bucket_proxy')
        )
        display(lifecycle_profile_df.head(20))


## Loader Smoke Test

여기서는 실제 학습 전에 각 모델이 어떤 `future_exo` shape를 받는지 확인합니다.

- `patchtst_exo`, `exotst`: `(B, H, E_future)`가 0보다 큰 shape여야 정상
- `timexer`: past-only 경로이므로 `(B, H, 0)`가 기대값


In [ ]:
def inspect_model_batches(model_names: list[str]) -> None:
    for spec in resolve_model_specs(model_names):
        future_cols = []
        effective_future_exo_cb = None
        effective_part_future_exo_fn = None
        if spec.use_future_exogenous:
            if future_exo_source == 'columns':
                future_cols = list(future_exo_cont_cols_effective)
            else:
                effective_future_exo_cb = future_exo_cb
                effective_part_future_exo_fn = part_future_exo_fn

        req = DataRequest(
            df=train_one_table,
            window=DataWindowConfig(
                lookback=lookback,
                horizon=horizon,
                freq=FREQ,
            ),
            columns=DataColumnConfig(
                id_col=ID_COL,
                date_col=DATE_COL,
                y_col=Y_COL,
            ),
            exogenous=ExogenousConfig(
                use_exogenous_mode=True,
                use_past_exogenous=True,
                use_future_exogenous=spec.use_future_exogenous,
                past_exo_cont_cols=list(past_exo_cont_cols_effective),
                past_exo_cat_cols=list(PAST_EXO_CAT_COLS),
                future_exo_cont_cols=future_cols,
                future_exo_cb=effective_future_exo_cb,
                part_future_exo_fn=effective_part_future_exo_fn,
            ),
            loader=LoaderConfig(
                batch_size=8,
                shuffle=False,
                num_workers=0,
                pin_memory=False,
                persistent_workers=False,
                prefetch_factor=2,
            ),
        )
        dm = build_datamodule(req)
        train_batch = next(iter(dm.get_train_loader(batch_size=8, shuffle=False, drop_last=False)))
        infer_batch = next(iter(dm.get_inference_loader_at_plan(int(plan_week))))
        print(
            f"{spec.label:14s} | train future_exo={tuple(train_batch[3].shape)} "
            f"| infer future_exo={tuple(infer_batch[3].shape)} "
            f"| past_exo={tuple(train_batch[4].shape)}"
        )

inspect_model_batches(models_to_run)


## Tiny AB Run Helpers

아래 셀은 notebook 안에서 바로 학습/추론을 돌릴 수 있도록 helper를 정의합니다.

기본값은 빠른 확인용이므로 epoch 수와 샘플 수를 작게 두었습니다. 시간이 오래 걸리면:
- `models_to_run`을 1~2개로 줄이기
- `sample_part_count` 줄이기
- 더 이른 `plan_week`로 옮기기


In [ ]:


arch_args = argparse.Namespace(
    patch_len=13,
    stride=6,
    patchtst_d_model=128,
    patchtst_layers=5,
    patchtst_d_ff=256,
    timexer_patch_len=13,
    timexer_d_model=128,
    timexer_heads=8,
    timexer_d_ff=256,
    timexer_e_layers=3,
    timexer_dropout=0.1,
    timexer_factor=5,
    timexer_activation='gelu',
    timexer_use_norm=True,
    exotst_d_model=128,
    exotst_heads=8,
    exotst_d_ff=256,
    exotst_dropout=0.1,
    exotst_attn_dropout=0.1,
    exotst_exo_enc_layers=2,
    exotst_fusion_layers=2,
    exotst_endo_dec_layers=2,
    exotst_exo_memory_mode='all',
    exotst_exo_nan_policy='zero+indicator',
    exotst_use_revin=True,
    exotst_subtract_last=True,
)
architecture = build_architecture_config(arch_args)

run_name = f"source_{future_exo_source}"
if use_lifecycle_future_features or use_lifecycle_past_features:
    run_name += f"__lifecycle_{lifecycle_start_mode}"
run_root = ensure_dir(artifact_root / run_name)


def build_notebook_data_request(
    *,
    df: pl.DataFrame,
    lookback: int,
    horizon: int,
    batch_size: int,
    num_workers: int,
    pin_memory: bool,
    persistent_workers: bool,
    prefetch_factor: int,
    shuffle: bool,
    use_future_exogenous: bool,
):
    future_cols = []
    effective_future_exo_cb = None
    effective_part_future_exo_fn = None
    if use_future_exogenous:
        if future_exo_source == 'columns':
            future_cols = list(future_exo_cont_cols_effective)
        else:
            if future_exo_cb is None or part_future_exo_fn is None:
                raise ValueError(
                    'Callback future exogenous mode requires both future_exo_cb and part_future_exo_fn.'
                )
            effective_future_exo_cb = future_exo_cb
            effective_part_future_exo_fn = part_future_exo_fn

    return DataRequest(
        df=df,
        window=DataWindowConfig(
            lookback=lookback,
            horizon=horizon,
            freq=FREQ,
        ),
        columns=DataColumnConfig(
            id_col=ID_COL,
            date_col=DATE_COL,
            y_col=Y_COL,
        ),
        exogenous=ExogenousConfig(
            use_exogenous_mode=True,
            use_past_exogenous=True,
            use_future_exogenous=use_future_exogenous,
            past_exo_cont_cols=list(past_exo_cont_cols_effective),
            past_exo_cat_cols=list(PAST_EXO_CAT_COLS),
            future_exo_cont_cols=future_cols,
            future_exo_cb=effective_future_exo_cb,
            part_future_exo_fn=effective_part_future_exo_fn,
        ),
        loader=LoaderConfig(
            batch_size=batch_size,
            shuffle=shuffle,
            num_workers=num_workers,
            pin_memory=pin_memory,
            persistent_workers=persistent_workers,
            prefetch_factor=prefetch_factor,
        ),
    )


def run_single_model_ab(spec):
    model_dir = ensure_dir(run_root / spec.label)

    train_req = build_notebook_data_request(
        df=train_one_table,
        lookback=lookback,
        horizon=horizon,
        batch_size=train_batch_size,
        num_workers=num_workers,
        pin_memory=pin_memory,
        persistent_workers=persistent_workers,
        prefetch_factor=prefetch_factor,
        shuffle=True,
        use_future_exogenous=spec.use_future_exogenous,
    )

    train_result = train(
        TrainRequest(
            data=train_req,
            freq=FREQ,
            models=[spec.request_key],
            architecture=architecture,
            trainer=TrainerConfig(
                warmup_epochs=warmup_epochs,
                spike_epochs=spike_epochs,
                lr=lr,
                use_intermittent=use_intermittent,
                val_use_weights=val_use_weights,
            ),
            ssl=SSLConfig(
                mode=ssl_mode if spec.request_key.startswith('patchtst') else 'off',
                pretrain_epochs=ssl_pretrain_epochs,
                mask_ratio=ssl_mask_ratio,
                loss_type=ssl_loss_type,
            ),
            runtime=RuntimeConfig(device=device),
            artifacts=ArtifactConfig(
                save_dir=str(model_dir),
                auto_save_dir=False,
            ),
        )
    )

    ckpt_path = train_result.primary_ckpt_path or train_result.ckpt_paths.get(spec.request_key)
    if not ckpt_path:
        raise RuntimeError(f'No checkpoint produced for {spec.label}')

    predictor = load_predictor(
        ckpt_path,
        device=device,
        forecaster_kwargs={
            'target_channel': 0,
            'fill_mode': 'copy_last',
            'use_winsor': True,
            'use_multi_guard': True,
        },
    )

    infer_req = build_notebook_data_request(
        df=infer_one_table,
        lookback=lookback,
        horizon=horizon,
        batch_size=infer_batch_size,
        num_workers=num_workers,
        pin_memory=pin_memory,
        persistent_workers=persistent_workers,
        prefetch_factor=prefetch_factor,
        shuffle=False,
        use_future_exogenous=spec.use_future_exogenous,
    )
    infer_dm = build_datamodule(infer_req)

    infer_loader = infer_dm.get_inference_loader_at_plan(int(plan_week))
    forecast_df = make_point_forecast_result_table(
        inference_loader=infer_loader,
        predictor=predictor,
        model_name=spec.label,
        plan_week=int(plan_week),
        horizon=horizon,
        device=device,
        max_parts=max_parts_per_plan,
    )
    forecast_df = attach_actuals(
        forecast_df,
        eval_target_df,
        id_col=ID_COL,
        date_col=DATE_COL,
        y_col=Y_COL,
    )
    return train_result, forecast_df


## Tiny AB Run

이 셀은 실제 학습과 추론을 수행합니다. 시간이 오래 걸리면 위 Config에서 `sample_part_count`, `models_to_run`을 줄이거나 더 이른 `plan_week`를 사용하세요.

In [ ]:
results = {}
forecast_tables = []

for spec in resolve_model_specs(models_to_run):
    print(f'=== RUN {spec.label} ({spec.request_key}) ===')
    train_result, forecast_df = run_single_model_ab(spec)
    results[spec.label] = train_result
    forecast_tables.append(forecast_df)
    print(spec.label, 'forecast shape:', forecast_df.shape)

combined_forecast_df = pl.concat(forecast_tables) if forecast_tables else pl.DataFrame()
combined_forecast_df


## Metrics and Plots

이 셀은 metric 테이블과 plot 파일을 생성하고, notebook 안에서도 바로 보여줍니다.

- `overall_df`: 모든 revision을 포함한 모델별 요약
- `latest_summary_df`: latest revision 기준 모델별 요약
- `latest_df`: part / forecast_week 기준 최신 revision 예측 long table


In [ ]:
overall_df, by_horizon_df, latest_summary_df = compute_metric_tables(combined_forecast_df)
latest_df = select_latest_revision(combined_forecast_df.filter(pl.col('actual').is_not_null()))

sampled_parts = select_plot_parts(latest_df, plot_part_count=plot_part_count)

plot_dir = ensure_dir(run_root / 'plots')
metric_plot_df = latest_summary_df if latest_summary_df.height > 0 else overall_df
metric_plot_path = plot_dir / 'metric_summary.png'
aggregate_plot_path = plot_dir / 'aggregate_latest_revision.png'
part_grid_plot_path = plot_dir / 'parts_latest_revision.png'

plot_metric_summary(metric_plot_df, metric_plot_path)
plot_latest_revision_aggregate(latest_df, aggregate_plot_path)
plot_latest_revision_part_grid(
    latest_df,
    sampled_parts=sampled_parts,
    save_path=part_grid_plot_path,
    ncols=3,
)

display(latest_summary_df if latest_summary_df.height > 0 else overall_df)
display(by_horizon_df.head(20))

if metric_plot_path.exists():
    display(Image(filename=str(metric_plot_path)))
if aggregate_plot_path.exists():
    display(Image(filename=str(aggregate_plot_path)))
if part_grid_plot_path.exists():
    display(Image(filename=str(part_grid_plot_path)))


## Sanity Checks

이 섹션은 아래를 빠르게 확인하기 위한 진단 코드입니다.

- 현재 결과 기준으로 모델별 `pred_sum / actual_sum` 비율 확인
- 같은 checkpoint로 `guarded` 예측과 `raw` 예측을 다시 비교
- underforecast가 학습 문제인지, guard/winsor 후처리 문제인지 분리 확인

In [ ]:
import matplotlib.pyplot as plt


def get_spec_by_label(label: str):
    spec_map = {spec.label: spec for spec in resolve_model_specs(models_to_run)}
    if label not in spec_map:
        raise KeyError(f"Unknown model label: {label}. available={list(spec_map)}")
    return spec_map[label]


def get_ckpt_path_for_label(label: str) -> str:
    spec = get_spec_by_label(label)
    train_result = results[label]
    ckpt_path = train_result.primary_ckpt_path or train_result.ckpt_paths.get(spec.request_key)
    if not ckpt_path:
        raise RuntimeError(f"No checkpoint found for {label}")
    return ckpt_path


def summarize_scale(df: pl.DataFrame, group_cols: list[str]) -> pl.DataFrame:
    if df.height == 0:
        return pl.DataFrame()

    valid = df.filter(pl.col("actual").is_not_null())
    if valid.height == 0:
        return pl.DataFrame()

    return (
        valid.group_by(group_cols)
        .agg(
            pl.len().alias("n_rows"),
            pl.col("prediction").mean().alias("pred_mean"),
            pl.col("prediction").median().alias("pred_p50"),
            pl.col("prediction").max().alias("pred_max"),
            pl.col("actual").mean().alias("actual_mean"),
            pl.col("actual").median().alias("actual_p50"),
            pl.col("actual").max().alias("actual_max"),
            pl.col("prediction").sum().alias("pred_sum"),
            pl.col("actual").sum().alias("actual_sum"),
        )
        .with_columns(
            pl.when(pl.col("actual_sum").abs() > 1e-12)
            .then(pl.col("pred_sum") / pl.col("actual_sum"))
            .otherwise(None)
            .alias("pred_to_actual_ratio")
        )
        .sort(group_cols)
    )


def rerun_forecast_with_guard_settings(
    label: str,
    *,
    use_winsor: bool,
    use_multi_guard: bool,
    max_parts: int | None = None,
) -> pl.DataFrame:
    spec = get_spec_by_label(label)
    ckpt_path = get_ckpt_path_for_label(label)

    predictor = load_predictor(
        ckpt_path,
        device=device,
        forecaster_kwargs={
            "target_channel": 0,
            "fill_mode": "copy_last",
            "use_winsor": use_winsor,
            "use_multi_guard": use_multi_guard,
        },
    )

    infer_req = build_notebook_data_request(
        df=infer_one_table,
        lookback=lookback,
        horizon=horizon,
        batch_size=infer_batch_size,
        num_workers=num_workers,
        pin_memory=pin_memory,
        persistent_workers=persistent_workers,
        prefetch_factor=prefetch_factor,
        shuffle=False,
        use_future_exogenous=spec.use_future_exogenous,
    )
    infer_dm = build_datamodule(infer_req)
    infer_loader = infer_dm.get_inference_loader_at_plan(int(plan_week))

    forecast_df = make_point_forecast_result_table(
        inference_loader=infer_loader,
        predictor=predictor,
        model_name=label,
        plan_week=int(plan_week),
        horizon=horizon,
        device=device,
        max_parts=max_parts or max_parts_per_plan,
    )
    forecast_df = attach_actuals(
        forecast_df,
        eval_target_df,
        id_col=ID_COL,
        date_col=DATE_COL,
        y_col=Y_COL,
    )

    variant = f"winsor_{int(use_winsor)}__guard_{int(use_multi_guard)}"
    return forecast_df.with_columns(pl.lit(variant).alias("variant"))


In [ ]:
# 1) 현재 이미 만든 combined_forecast_df 기준으로
# 모델별 예측 스케일이 GT 대비 어느 수준인지 먼저 확인
current_latest_df = select_latest_revision(
    combined_forecast_df.filter(pl.col("actual").is_not_null())
)
current_scale_df = summarize_scale(current_latest_df, group_cols=["model_name"])
display(current_scale_df)


# 2) 한 모델을 골라 guarded / raw 예측을 다시 비교
# 필요하면 'timexer' 또는 'exotst'로 바꿔서 돌리면 됩니다.
diagnostic_model = "patchtst_exo" if "patchtst_exo" in models_to_run else models_to_run[0]
diagnostic_max_parts = min(256, max_parts_per_plan)

print(f"diagnostic_model={diagnostic_model}")
print(f"plan_week={plan_week}")
print(f"max_parts={diagnostic_max_parts}")
print(f"checkpoint={get_ckpt_path_for_label(diagnostic_model)}")

guarded_diag_df = rerun_forecast_with_guard_settings(
    diagnostic_model,
    use_winsor=True,
    use_multi_guard=True,
    max_parts=diagnostic_max_parts,
)

raw_diag_df = rerun_forecast_with_guard_settings(
    diagnostic_model,
    use_winsor=False,
    use_multi_guard=False,
    max_parts=diagnostic_max_parts,
)

diag_compare_df = pl.concat([guarded_diag_df, raw_diag_df], how="vertical_relaxed")
diag_latest_df = select_latest_revision(
    diag_compare_df.filter(pl.col("actual").is_not_null()),
    extra_group_cols=["variant"],
)

# variant별 전체 스케일 비교
diag_scale_df = summarize_scale(diag_latest_df, group_cols=["variant"])
display(diag_scale_df)

# forecast week별 aggregate 비교
diag_weekly_df = (
    diag_latest_df.group_by(["variant", "forecast_week"])
    .agg(
        pl.col("prediction").sum().alias("pred_sum"),
        pl.col("actual").sum().alias("actual_sum"),
    )
    .with_columns(
        pl.when(pl.col("actual_sum").abs() > 1e-12)
        .then(pl.col("pred_sum") / pl.col("actual_sum"))
        .otherwise(None)
        .alias("pred_to_actual_ratio")
    )
    .sort(["forecast_week", "variant"])
)
display(diag_weekly_df.head(30))

# 간단 플롯
first_variant = (
    diag_weekly_df.get_column("variant").unique().to_list()[0]
    if diag_weekly_df.height > 0
    else None
)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

if diag_weekly_df.height > 0:
    for variant in diag_weekly_df.get_column("variant").unique().to_list():
        sub = diag_weekly_df.filter(pl.col("variant") == variant).sort("forecast_week")
        axes[0].plot(
            sub.get_column("forecast_week").to_list(),
            sub.get_column("pred_sum").to_list(),
            marker="o",
            label=variant,
        )

    if first_variant is not None:
        gt_sub = diag_weekly_df.filter(pl.col("variant") == first_variant).sort("forecast_week")
        axes[0].plot(
            gt_sub.get_column("forecast_week").to_list(),
            gt_sub.get_column("actual_sum").to_list(),
            marker="o",
            linewidth=2.5,
            label="GT",
        )

    axes[0].set_title(f"{diagnostic_model}: latest revision aggregate")
    axes[0].set_xlabel("forecast_week")
    axes[0].set_ylabel("sum_qty")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    for variant in diag_weekly_df.get_column("variant").unique().to_list():
        sub = diag_weekly_df.filter(pl.col("variant") == variant).sort("forecast_week")
        axes[1].plot(
            sub.get_column("forecast_week").to_list(),
            sub.get_column("pred_to_actual_ratio").to_list(),
            marker="o",
            label=variant,
        )

    axes[1].axhline(1.0, color="white", linestyle="--", linewidth=1.5, alpha=0.7)
    axes[1].set_title(f"{diagnostic_model}: pred / actual ratio")
    axes[1].set_xlabel("forecast_week")
    axes[1].set_ylabel("pred_to_actual_ratio")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
else:
    axes[0].text(0.5, 0.5, "No diagnostic rows", ha="center", va="center")
    axes[1].text(0.5, 0.5, "No diagnostic rows", ha="center", va="center")

plt.tight_layout()
plt.show()
